# Warp resampling test-case inspector

Visual debugging aid for the warp golden-regression tests in `test/test_04_raster.py`
(`test_warp_resampling_golden`). When one of those tests fails, run this notebook to *see*
what changed: it builds the same deterministic source raster, warps it with the selected
algorithm, and shows the source, the produced output, the committed golden, and their
difference side by side.

Golden references live in `geokit/data/raster_results/warp_resampling_<alg>.tif`. They are
generated automatically the first time the test runs and committed to the repo. To regenerate
after a deliberate GDAL/PROJ upgrade, run the tests with `GEOKIT_REGEN_GOLDEN=1` and commit the
updated files.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

# Make the repo importable: walk up from the notebook directory until we find the geokit package.
REPO_ROOT = os.getcwd()
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, "geokit")):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

from geokit import raster
from test.helpers import make_resampling_test_raster, raster_result, assert_raster_equal

print("repo root:", REPO_ROOT)

## Pick a case

Set `RESAMPLE_ALG` to the algorithm you want to inspect. Options: `near`, `bilinear`, `cubic`,
`cubicspline`, `lanczos`, `average`, `rms`, `mode`, `max`, `min`, `med`, `q1`, `q3`, `sum`.

In [ ]:
RESAMPLE_ALG = "cubic"
SRC_PIXEL = 100  # source resolution (32x32 grid)
OUT_PIXEL = 200  # output resolution -> 2:1 downsample to 16x16

In [ ]:
def load_case(alg, src_pixel=SRC_PIXEL, out_pixel=OUT_PIXEL):
    """Return (source_matrix, produced_matrix, golden_matrix_or_None, golden_path)."""
    source = make_resampling_test_raster(pixel=src_pixel)
    produced = raster.warp(source, resampleAlg=alg, pixelWidth=out_pixel, pixelHeight=out_pixel)
    src_mat = raster.extractMatrix(source)
    out_mat = raster.extractMatrix(produced)
    golden_path = raster_result(f"warp_resampling_{alg}.tif")
    gold_mat = raster.extractMatrix(golden_path) if os.path.isfile(golden_path) else None
    return src_mat, out_mat, gold_mat, golden_path


src_mat, out_mat, gold_mat, golden_path = load_case(RESAMPLE_ALG)
print("source", src_mat.shape, "-> output", out_mat.shape)
print("golden:", golden_path, "(found)" if gold_mat is not None else "(MISSING - run pytest to generate)")

## Visualise source / output / golden / difference

In [ ]:
def show_case(alg):
    src_mat, out_mat, gold_mat, golden_path = load_case(alg)
    has_gold = gold_mat is not None
    ncols = 4 if has_gold else 2
    fig, axes = plt.subplots(1, ncols, figsize=(4.2 * ncols, 4))

    def panel(ax, mat, title, **kw):
        h = ax.imshow(mat, interpolation="nearest", **kw)
        ax.set_title(title)
        fig.colorbar(h, ax=ax, fraction=0.046, pad=0.04)

    panel(axes[0], src_mat, f"source {src_mat.shape}")
    panel(axes[1], out_mat, f"warp '{alg}' {out_mat.shape}")
    if has_gold:
        panel(axes[2], gold_mat, f"golden {gold_mat.shape}")
        diff = out_mat.astype(float) - gold_mat.astype(float)
        vmax = max(abs(diff).max(), 1e-9)
        panel(axes[3], diff, f"produced - golden (max|d|={abs(diff).max():.3g})", cmap="RdBu", vmin=-vmax, vmax=vmax)

    fig.suptitle(f"resampleAlg = '{alg}'")
    plt.tight_layout()
    plt.show()

    if has_gold:
        produced = raster.warp(
            make_resampling_test_raster(pixel=SRC_PIXEL), resampleAlg=alg, pixelWidth=OUT_PIXEL, pixelHeight=OUT_PIXEL
        )
        try:
            assert_raster_equal(golden_path, produced)
            print(f"OK: '{alg}' matches the committed golden.")
        except AssertionError as exc:
            print(f"MISMATCH for '{alg}': {exc}")


show_case(RESAMPLE_ALG)

## Compare all algorithms at a glance

In [ ]:
ALGS = [
    "near",
    "bilinear",
    "cubic",
    "cubicspline",
    "lanczos",
    "average",
    "rms",
    "mode",
    "max",
    "min",
    "med",
    "q1",
    "q3",
    "sum",
]

fig, axes = plt.subplots(2, 7, figsize=(20, 6))
for ax, alg in zip(axes.ravel(), ALGS):
    _, out_mat, _, _ = load_case(alg)
    h = ax.imshow(out_mat, interpolation="nearest")
    ax.set_title(alg)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()